In [ ]:

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply() 

In [ ]:
#Import libraries, APIs and LLM
from crewai import Agent, Crew, Task, Process


In [ ]:
# lets set up the keys and the model we are gonna be using 
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"
os.environ["SERPER_API_KEY"] = get_serper_api_key()

In [ ]:
'''# lets set up the keys and the model we are gonna be using 
import os
from utils import get_gemini_api_key, get_serper_api_key

gemini_api_key = get_gemini_api_key()
os.environ["GEMINI_MODEL_NAME"] = "models/gemini-2.5-flash-lite-preview-06-17"
os.environ["SERPER_API_KEY"] =get_serper_api_key()
'''

In [ ]:
# We go ahead and import our tools, we gonna be using two tools in this case, the Scrape WebsiteTool and SerperDecTool
# tgink about how this tools will fit together, The Serp allows ---- and then the Scrap ---
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

# Initialize the tools
search_tool = SerperDevTool()    # this allows the agent to search the google and then get the result back
scrape_tool = ScrapeWebsiteTool()   # this allows the agent to go into those websites that it finds and get in their content so that you can use that during the execution 

In [ ]:
# Our Agent 1:is Venue Coordinator, this agent has one single goal
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="Identify and book an appropriate venue "
    "based on event requirements",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "With a keen sense of space and "
        "understanding of event logistics, "
        "you excel at finding and securing "
        "the perfect venue that fits the event's theme, "
        "size, and budget constraints."
    )
)

In [ ]:
 # Our Agent 2: is Logistics Manager,  this agent make sure to think through the logistic of the event. now the first agent already found the few options for the venue, 
# the second agent is going to think about how would this workout and making sure the venue is appropraite for us. so lets go ahead and create this agent as well
logistics_manager = Agent(
    role='Logistics Manager',
    goal=(
        "Manage all logistics for the event "
        "including catering and equipment"
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Organized and detail-oriented, "
        "you ensure that every logistical aspect of the event "
        "from catering to equipment setup "
        "is flawlessly executed to create a seamless experience."
    )
)

In [ ]:
# our 3 Agent is Marketing, its suppose to work on top of the venue and logistics and figure it out 
# how can we market this even, so that we can get as many people showing up today. so this agent is gonna be creative ---- so let go ahead and create that agent as well
marketing_communications_agent = Agent(
    role="Marketing and Communications Agent",
    goal="Effectively market the event and "
         "communicate with participants",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Creative and communicative, "
        "you craft compelling messages and "
        "engage with potential attendees "
        "to maximize event exposure and participation."
    )
)

In [ ]:
# you can see how this can be a pretty intresting tool, we have a system now that is not only finding things about the event
# but thinking about logistics and also thinking about marketing, everything in one go. this would be extremely hard to pull it off with conventional 
# programming if you are not building an AI application. so now that we have this agents, we are gonna create a pydantic object that is going to hold our 
# venue details, this is intresting because its gonna show up a new feature for task that we havent discause yet

In [ ]:
# so in here you can see that we are importing the pydentic base model and creating a class that is going to inherite the vanue details
# which have these qttributes, name ---
# the reason why we are creating this is bcos so that our agent can work with it and  populate instance of this as they work
from pydantic import BaseModel
# Define a Pydantic model for venue details .. pydentic is a library athat allows you to create model in a simple way
# (demonstrating Output as Pydantic)
class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: int
    booking_status: str

In [ ]:
# so let go ahead and create our first task. this task is responsable to find a venue. so you can seee
# that it expect few input here, event_city and event_topic and here we are using other attributes that we havent used before like output_json and we 
# passing our pydentic object (VenueDetails) and then output that result into a jsonfile that we can actually use
venue_task = Task(
    description="Find a venue in {event_city} "
                "that meets criteria for {event_topic}.",
    expected_output="All the details of a specifically chosen"
                    "venue you found to accommodate the event.",
    human_input=True,
    output_json=VenueDetails,
    output_file="venue_details.json",  
      # Outputs the venue details as a JSON file
    agent=venue_coordinator
)

In [ ]:
# so now lets create ouw second task, this task is responsiablle for coordinating catering, equip.
# and its ecpectiong a few diff variables, looking for what is number of participant and what is the tentative date for this evnt to happen
# you can see that we are having diff attributes, the human_input, this means that  before this task is completed, the agent is going to stop and ask us 
# the operator if we want to give an input, if we like or wanted to change anything
# we are also doing async_execution, which means that this task is going to be excuting in parallel with any other tasks that come after that
# this ask depend s on the vanue task but it does nothig to do with the final task
logistics_task = Task(
    description="Coordinate catering and "
                 "equipment for an event "
                 "with {expected_participants} participants "
                 "on {tentative_date}.",
    expected_output="Confirmation of all logistics arrangements "
                    "including catering and equipment setup.",
    human_input=True,
    async_execution=True,
    agent=logistics_manager
)

In [ ]:
# a final task is gonna be a marketing task is the task responsable for creating a marketing campain that would help us prompt this event
# you can here that again we are taking about the event_topic, expected participant and we are gonna report on marketing activities
# we can also see that with are output file in this case a mark down file 
marketing_task = Task(
    description="Promote the {event_topic} "
                "aiming to engage at least"
                "{expected_participants} potential attendees.",
    expected_output="Report on marketing activities "
                    "and attendee engagement formatted as markdown.",
    async_execution=True,
    output_file="marketing_report.md",  # Outputs the report as a text file
    agent=marketing_communications_agent
)

In [ ]:
event_management_crew = Crew(
    agents=[venue_coordinator, logistics_manager],
    tasks=[venue_task, logistics_task],   # logistics is async and LAST → OK
    verbose=True
)

In [ ]:
marketing_crew = Crew(
    agents=[marketing_communications_agent],
    tasks=[marketing_task],  # async allowed because it's the only one
    verbose=True
)

In [ ]:
# so now that we have our tasks lets create our crew which is pretty straigth forward
# Define the crew with agents and tasks
# Define the crew with agents and tasks
'''event_management_crew = Crew(
    agents=[venue_coordinator, 
            logistics_manager, 
            marketing_communications_agent],
    
    tasks=[venue_task, 
           logistics_task, 
           marketing_task],
    
    verbose=True,

)
'''

In [ ]:
# lets keep in mind that there are lots of variable that we used through out the task and the agent that needs to be enterpulated, 
# so lets start by setting those up.
# thes are all the input that this crew need in other for it o do its research to actually fullfil its task
event_details = {
    'event_topic': "AI IN ACTION",
    'event_description': "A gathering of tech innovators "
                         "and industry leaders "
                         "to explore future technologies.",
    'event_city': "Edinburgh City",
    'tentative_date': "2025-11-21",
    'expected_participants': 100,
    'budget': 600,
    'venue_type': "Lister Learning and Teaching Center, 5 Roxburgh pl, Edinburgh EH8 9SU"
}

In [ ]:
async def run_parallel():
    results = await asyncio.gather(
        event_management_crew.kickoff_async(inputs=event_details),
        marketing_crew.kickoff_async(inputs=event_details)
    )
    return results

results = asyncio.run(run_parallel())
print(results)

In [ ]:
# so lets kick it off and see how that goes
#result = event_management_crew.kickoff(inputs=event_details)

In [ ]:
# you can see from the get go that the venue coordinator kick thins off by trying to find the vanue in Edinburgh City
# that where we want this event to happen
# you can see that it decides to search the internet for AI in Action in Edinburgh. so it found a bouch of result as we can see 
# and again it decide to scrap one of this results (see Thought)
# it quit large but it found these final result and wanted to sat whats your feedback... i will keep it simple by saying yes, i like this option
# if you look at what ahappend its gonna pass on to the next agent that is going to figure it out not only the cetering but also the marketing
# you can see the final answer and look how cool it that, the final answer is now in json not task any longer
# you save this into a database.

#while ohe agent is looking into the catering, the other is looking into how to promot an event like this 
# end this is the json file, the venue, capacity etc and we can look at what the marketing campain including the digital, user bugdget etc...

# think about how amzing it is, we basically created a multi agent system that is capable of not only finding a good venue, but organizing and marketing
# strategis in planning it out for you, to act on. this is just tip of an eyes beg what you can achived with multi agent system

In [ ]:
import json
from pprint import pprint

with open('venue_details.json') as f:
   data = json.load(f)

pprint(data)

In [ ]:
from IPython.display import Markdown
Markdown("marketing_report.md")